In [ ]:
"""
Parte4_peerweights.py -- 4.4: quem sao as empresas comparaveis segundo o modelo (Geertsema & Lu, Apendice A.3).

Com perda L2, sem regularizacao e sem bagging, cada folha da arvore m vale  eta * media dos residuos do treino
que cairam nela. Como os residuos sao lineares nos alvos de treino y, a previsao tambem e:
        F_M(x) = w(x)' y,      com  w_0 = (1/n) 1,
        w_m(x) = w_(m-1)(x) + eta * L_m(x) (I - W_(m-1))
onde L_m(x) e o vetor com 1/n_folha nas empresas de treino que estao na mesma folha de x na arvore m, e
W_(m-1) e a matriz n x n dos pesos das proprias empresas de treino. Como as linhas de L somam 1 e as de
(I - W) somam 0, os pesos de cada empresa SOMAM 1 em todas as iteracoes (podem ser negativos).
Verificacao obrigatoria: w(x)' y == previsao do LightGBM (tolerancia TOL_SOMA).
Saidas em resultados_parte4/: peers_caso_<ticker>_<alvo>_<AAAAMM>.csv, peers_resumo.csv
"""
import numpy as np
import pandas as pd

import comum_parte4 as C

TOL_SOMA = 1e-5          # LightGBM acumula em float32/float64 misto; erros ~1e-7 sao normais
CASO = None              # ex.: ("ABCD3", "lnm2b", "2019-12-31"); None escolhe uma empresa do ultimo mes
MESES_RESUMO = 12        # resumo dos pesos nos ultimos N meses (None = todos)


def media_por_folha(M, folhas):
    """Para cada linha i, media das linhas de M que estao na mesma folha de i (operador L aplicado a M)."""
    _, inv = np.unique(folhas, return_inverse=True)
    soma = np.zeros((inv.max() + 1, M.shape[1]))
    np.add.at(soma, inv, M)
    cont = np.bincount(inv).astype(float)
    return soma[inv] / cont[inv, None]


def peer_weights(booster, X_treino, X_alvo):
    """Matriz de pesos (n_alvo x n_treino). Linhas de X_alvo podem ser de teste ou de treino."""
    checa_parametros(booster)
    eta = C.taxa_aprendizado(booster)
    n = len(X_treino)
    folhas_tr = booster.predict(X_treino, pred_leaf=True).reshape(n, -1)
    folhas_al = booster.predict(X_alvo, pred_leaf=True).reshape(len(X_alvo), -1)
    W = np.full((n, n), 1.0 / n)                 # pesos das empresas de treino nelas mesmas
    w = np.full((len(X_alvo), n), 1.0 / n)       # pesos das empresas avaliadas
    I = np.eye(n)
    for m in range(folhas_tr.shape[1]):
        R = I - W                                # residuo de cada empresa de treino como combinacao de y
        # pesos do alvo: media de R nas empresas de treino da mesma folha
        ft, fa = folhas_tr[:, m], folhas_al[:, m]
        _, inv = np.unique(np.concatenate([ft, fa]), return_inverse=True)
        it, ia = inv[:n], inv[n:]
        soma = np.zeros((inv.max() + 1, n))
        np.add.at(soma, it, R)
        cont = np.bincount(it, minlength=inv.max() + 1).astype(float)
        if (cont[ia] == 0).any():
            raise RuntimeError("empresa avaliada caiu em folha sem empresas de treino: X_treino nao e o treino do modelo")
        w = w + eta * soma[ia] / cont[ia, None]
        W = W + eta * media_por_folha(R, ft)
    return w


def checa_parametros(booster):
    p = C.parametros_gravados(booster)
    problemas = []
    if p.get("objective", "regression").split()[0] not in ("regression", "regression_l2", "l2", "mse"):
        problemas.append(f"objective={p.get('objective')}")
    for k in ("lambda_l1", "lambda_l2", "path_smooth"):
        if float(p.get(k, 0) or 0) != 0:
            problemas.append(f"{k}={p[k]}")
    if float(p.get("bagging_fraction", 1) or 1) < 1 and int(p.get("bagging_freq", 0) or 0) > 0:
        problemas.append("bagging ligado")
    if p.get("data_sample_strategy", "bagging") == "goss" or "goss" in p.get("boosting", ""):
        problemas.append("GOSS")
    if problemas:
        raise ValueError(f"peer weights exatos exigem L2 sem regularizacao/amostragem; modelo tem: {problemas}")


def verifica(w, y, previsao):
    desvio = np.abs(w @ y - previsao).max()
    if desvio > TOL_SOMA:
        raise AssertionError(f"soma(peso x alvo) difere da previsao do GBM em {desvio:.2e}")
    if np.abs(w.sum(1) - 1).max() > 1e-8:
        raise AssertionError("pesos nao somam 1")
    return desvio


def pesos_do_modelo(painel, prev, alvo, data, bloco):
    booster, X_tr, y_tr, X_te, treino, teste = C.dados_do_modelo(painel, prev, alvo, data, bloco)
    w = peer_weights(booster, X_tr, X_te)
    desvio = verifica(w, y_tr, teste["pred_log"].to_numpy(float))
    return w, treino.reset_index(drop=True), teste.reset_index(drop=True), y_tr, desvio


def caso_ilustrativo(painel, prev, ticker, alvo, data):
    data = pd.Timestamp(data) + pd.offsets.MonthEnd(0)
    linha = prev[(prev["ticker"] == ticker) & (prev["alvo"] == alvo) & (prev["data"] == data)]
    if linha.empty:
        raise KeyError(f"{ticker} {alvo} {data:%Y-%m} sem previsao")
    bloco = int(linha["bloco"].iloc[0])
    w, treino, teste, y_tr, desvio = pesos_do_modelo(painel, prev, alvo, data, bloco)
    i = int(np.flatnonzero(teste["ticker"].to_numpy() == ticker)[0])
    pm = painel[painel["data"] == data].set_index("ticker")
    tab = treino[["ticker"]].assign(peso=w[i], alvo_log=y_tr, multiplo=np.exp(y_tr))
    if tem_setor(painel):
        tab["setor"] = pm.loc[tab["ticker"], "setor"].to_numpy()
    tab = tab.sort_values("peso", ascending=False).reset_index(drop=True)
    tab["peso_acumulado"] = tab["peso"].cumsum()
    print(f"\nCaso: {ticker} | {alvo} | {data:%Y-%m} | bloco {bloco} | n_treino={len(tab)}")
    print(f"previsao GBM = {linha['pred_log'].iloc[0]:.6f} | soma(peso x alvo) = {w[i] @ y_tr:.6f} | desvio {desvio:.1e}")
    print(f"soma dos pesos = {tab['peso'].sum():.6f} | n peers efetivos (1/sum w^2) = {1 / (tab['peso'] ** 2).sum():.1f}"
          f" | pesos negativos: {(tab['peso'] < 0).sum()}")
    if "setor" in tab:
        s = pm.loc[ticker, "setor"]
        mesmo = tab["setor"].eq(s)
        print(f"setor {s}: {mesmo.mean():.0%} das empresas de treino, {tab.loc[mesmo, 'peso'].sum():.0%} do peso total, "
              f"{mesmo.head(10).sum()} dos 10 maiores peers")
    print("\n10 mais comparaveis:\n", tab.head(10).round(4).to_string(index=False))
    print("\n10 menos comparaveis:\n", tab.tail(10).round(4).to_string(index=False))
    return tab


def tem_setor(painel):
    return "setor" in painel and painel["setor"].notna().any()


def resumo(painel, prev, meses=MESES_RESUMO):
    """Concentracao e afinidade setorial dos pesos em todos os modelos dos ultimos `meses` meses."""
    datas = sorted(prev["data"].unique())
    datas = datas[-meses:] if meses else datas
    linhas = []
    pm_setor = painel.set_index(["data", "ticker"])["setor"] if tem_setor(painel) else None
    for (alvo, data, bloco), _ in prev[prev["data"].isin(datas)].groupby(["alvo", "data", "bloco"]):
        w, treino, teste, _, desvio = pesos_do_modelo(painel, prev, alvo, data, bloco)
        ws = -np.sort(-w, axis=1)
        lin = dict(alvo=alvo, data=data, bloco=bloco, n_teste=len(teste), n_treino=len(treino), desvio_max=desvio,
                   n_efetivo=(1 / (w ** 2).sum(1)).mean(), peso_top10=ws[:, :10].sum(1).mean(),
                   frac_pesos_neg=(w < 0).mean())
        if pm_setor is not None:
            st = pm_setor.loc[data].loc[treino["ticker"]].to_numpy()
            se = pm_setor.loc[data].loc[teste["ticker"]].to_numpy()
            mesmo = se[:, None] == st[None, :]
            lin["peso_mesmo_setor"] = (w * mesmo).sum(1).mean()
            lin["frac_treino_mesmo_setor"] = mesmo.mean(1).mean()       # o que se esperaria com pesos iguais
        linhas.append(lin)
    return pd.DataFrame(linhas)


def main():
    C.PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    painel, prev = C.ler_painel(), C.ler_previsoes()
    if CASO is None:
        alvo = "lnm2b" if (prev["alvo"] == "lnm2b").any() else prev["alvo"].iloc[0]
        ult = prev[(prev["alvo"] == alvo)]
        ult = ult[ult["data"] == ult["data"].max()]
        caso = (ult.sort_values("valor_mercado").iloc[len(ult) // 2]["ticker"], alvo, ult["data"].iloc[0])
    else:
        caso = CASO
    tab = caso_ilustrativo(painel, prev, *caso)
    tab.to_csv(C.PASTA_SAIDA / f"peers_caso_{caso[0]}_{caso[1]}_{pd.Timestamp(caso[2]):%Y%m}.csv", index=False)
    r = resumo(painel, prev)
    r.to_csv(C.PASTA_SAIDA / "peers_resumo.csv", index=False)
    print(f"\nResumo ({r['data'].nunique()} meses, {len(r)} modelos): desvio maximo da verificacao {r['desvio_max'].max():.1e}")
    cols = [c for c in ["n_treino", "n_efetivo", "peso_top10", "frac_pesos_neg", "peso_mesmo_setor",
                        "frac_treino_mesmo_setor"] if c in r]
    print(r.groupby("alvo")[cols].mean().round(3).to_string())


if __name__ == "__main__":
    main()